# Arm G dose response — does over-removal flip the decision?

Run 1 (seed 108) removed the rank-1 layer-16 conflict direction from every layer and
every token position: 33% of the condition contrast gone, coherence intact, and the
deciding rows driven from a minimum margin of +2.125 down to **+0.125** — about 95% of
the way to flipping, and stopping just short. Zero flips was a threshold artifact, not
an inert variable.

This run scales the removal past unit strength: `x ← x − k·(x·r)·r`, all layers, all
positions, k ∈ {0.5, 1, 1.5, 2, 3, 4}. k=1 is the run-1 condition; k=2 reflects the
component through zero.

**Frozen primary:** does some dose drive the conflict-condition decline rate below 0.25
(baseline 0.50) while the model stays coherent *and* while matched-dose random
directions do not?

**Why matched-dose controls matter here:** a flip is the outcome I expect, which is
exactly when it should be hardest to earn. Four random directions orthogonal to the
conflict direction are carried through every dose identically. If they flip too at the
same k, the result is about perturbation size, not about the variable.

Greedy continuations are sampled at every dose, so "coherent" can be read as text
rather than only as a statistic.

Set **Runtime → Change runtime type → A100 GPU**. ~250 batched passes plus short
generations; a few minutes after model load.

In [ ]:
# Colab supplies torch/CUDA.
print("Protocol: ARM_G_DOSE_V1")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/arm-g-dose-launch"
LAUNCH_FILES = (
    "arm_g_dose.py",
    "arm_g_cross_layer.py",
    "arm_g_causal_subspace.py",
    "arm_g_causal_dose_ablation.py",
    "arm_g_causal.py",
    "arm_g_phase1.py",
    "arm_g_scenarios.py",
)
for name in LAUNCH_FILES:
    source = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(source), f"Missing {source}"
    shutil.copy2(source, f"/content/{name}")
print("Arm G dose launch files staged: OK")

In [ ]:
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/arm-g-dose-seed109-v1"
PAIRS_PER_FAMILY = 16
BOOTSTRAP = 2000
RANDOM_DIRECTIONS = 4
BATCH_SIZE = 16

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

In [ ]:
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Deterministic tests: dose scaling algebra and all three decision branches.
import os, subprocess, sys
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
base_cmd = [
    sys.executable, "/content/arm_g_dose.py",
    "--model", ACTING_MODEL,
    "--output-dir", WORK_DIR,
    "--pairs-per-family", str(PAIRS_PER_FAMILY),
    "--bootstrap", str(BOOTSTRAP),
    "--random-directions", str(RANDOM_DIRECTIONS),
    "--batch-size", str(BATCH_SIZE),
]
subprocess.run(base_cmd + ["--self-test"], check=True, env=env)

In [ ]:
# Dose sweep with matched-dose random controls and generation probes.
subprocess.run(base_cmd, check=True, env=env)

In [ ]:
import json
result_path = f"{WORK_DIR}/arm_g_dose_result.json"
result = json.load(open(result_path))
summary = {
    "decision": result["decision"],
    "decision_reasons": result["decision_reasons"],
    "decision_detail": result["decision_detail"],
    "baseline": result["baseline"],
    "dose_response": {
        k: {
            "attenuation": v["attenuation"]["overall"],
            "attenuation_fraction": v["attenuation_fraction"],
            "conflict_decline_rate": v["conflict_decline_rate"],
            "reachable_decline_rate": v["reachable_decline_rate"],
            "decision_flips": v["decision_flips"],
            "conflict_row_mean_shift": v["conflict_row_mean_shift"],
            "reachable_row_mean_shift": v["reachable_row_mean_shift"],
            "coherence": v["coherence"],
        }
        for k, v in result["dose_response"].items()
    },
    "random_controls_by_dose": result["random_controls_by_dose"],
    "sample_counts": result["sample_counts"],
}
print(json.dumps(summary, indent=2))
print("\n=== generations ===")
for key in result["generations"]:
    texts = [g["continuation"].replace("\n", " ")[:70] for g in result["generations"][key][:3]]
    print(f"{key:>12}: {texts}")

In [ ]:
import base64, gzip, json

summary_path = f"{WORK_DIR}/arm_g_dose_result_summary.json"
archive_path = f"{WORK_DIR}/arm_g_dose_result.json.gz.b64"
with open(summary_path, "w") as h:
    json.dump({**summary, "generations": result["generations"],
        "protocol": {"source_seeds": [101, 102], "evaluation_seed": 109,
        "direction_layer": 16, "doses": [0.5, 1.0, 1.5, 2.0, 3.0, 4.0],
        "bootstrap_repetitions": BOOTSTRAP},
        "full_result_artifact": "arm_g_dose_result.json.gz.b64"}, h, indent=1)
raw = json.dumps(result).encode("utf-8")
with open(archive_path, "wb") as h:
    h.write(base64.b64encode(gzip.compress(raw)))
print("summary:", summary_path)
print("archive:", archive_path)